# Exercise 1: Text Classification with RNN

## Task 1: Load the IMDB Dataset and Select a Sample

In [2]:

import pandas as pd

# Load the IMDB dataset
df = pd.read_csv("D:\Projects\GenAI\IMDB Dataset.csv")

# Display basic information
print("Original dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())

# Check sentiment distribution
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

# Select 2,000 reviews for faster training
df = df.sample(n=2000, random_state=42).reset_index(drop=True)

# Display the sample
print("\nSample dataset shape:", df.shape)
display(df.head())


<>:4: SyntaxWarning: invalid escape sequence '\P'
<>:4: SyntaxWarning: invalid escape sequence '\P'
C:\Users\prana\AppData\Local\Temp\ipykernel_25252\3179546220.py:4: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("D:\Projects\GenAI\IMDB Dataset.csv")


Original dataset shape: (50000, 2)

Column names: ['review', 'sentiment']

Sentiment distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample dataset shape: (2000, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative



## Task 2: Convert Sentiment Labels to Numeric Values


In [3]:

# Convert sentiment labels into numeric values
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# Display the converted labels
print(df[["sentiment", "label"]].head())

# Check the label counts
print("\nNumeric label distribution:")
print(df["label"].value_counts())


  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0

Numeric label distribution:
label
0    1024
1     976
Name: count, dtype: int64



## Task 3: Split the Dataset into Training and Test Sets


In [4]:

from sklearn.model_selection import train_test_split

# Separate review text and numeric labels
X = df["review"]
y = df["label"]

# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display the sizes
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())


Training reviews: 1600
Testing reviews: 400

Training label distribution:
label
0    819
1    781
Name: count, dtype: int64

Testing label distribution:
label
0    205
1    195
Name: count, dtype: int64



## Task 4: Tokenize Reviews Using tiktoken


In [5]:

import tiktoken

# Load the required encoding
encoding = tiktoken.get_encoding("cl100k_base")

# Convert review text into token IDs
X_train_tokens = [
    encoding.encode(review, disallowed_special=())
    for review in X_train
]

X_test_tokens = [
    encoding.encode(review, disallowed_special=())
    for review in X_test
]

# Display sample token IDs
print("First review:")
print(X_train.iloc[0][:200])

print("\nFirst 20 token IDs:")
print(X_train_tokens[0][:20])

print("\nTotal training reviews tokenized:", len(X_train_tokens))
print("Total testing reviews tokenized:", len(X_test_tokens))


First review:
The story of Ed Gein is interesting, VERY interesting. This movie, however, interesting only in the fact that it was actually made. Kane Hodder's portrayal of Ed Gein is so far off, it's not even the 

First 20 token IDs:
[791, 3446, 315, 3279, 4323, 258, 374, 7185, 11, 48174, 7185, 13, 1115, 5818, 11, 4869, 11, 7185, 1193, 304]

Total training reviews tokenized: 1600
Total testing reviews tokenized: 400



## Task 5: Pad or Truncate Token Sequences


In [6]:

import torch
from torch.nn.utils.rnn import pad_sequence

# Set the fixed review length
MAX_LENGTH = 200

# Pad or truncate each review
def pad_or_truncate(token_sequences, max_length):
    padded_sequences = []

    for tokens in token_sequences:
        tokens = tokens[:max_length]
        tokens = tokens + [0] * (max_length - len(tokens))
        padded_sequences.append(tokens)

    return padded_sequences

X_train_padded = pad_or_truncate(X_train_tokens, MAX_LENGTH)
X_test_padded = pad_or_truncate(X_test_tokens, MAX_LENGTH)

# Check the results
print("Training data shape:", len(X_train_padded), "x", len(X_train_padded[0]))
print("Testing data shape:", len(X_test_padded), "x", len(X_test_padded[0]))

print("\nFirst review token count after padding:", len(X_train_padded[0]))


Training data shape: 1600 x 200
Testing data shape: 400 x 200

First review token count after padding: 200



## Task 6: Convert Token Sequences and Labels into Tensors


In [7]:

# Convert padded sequences into PyTorch tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

# Convert labels into PyTorch tensors
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

# Check tensor shapes and data types
print("Training input shape:", X_train_tensor.shape)
print("Testing input shape:", X_test_tensor.shape)

print("Training labels shape:", y_train_tensor.shape)
print("Testing labels shape:", y_test_tensor.shape)

print("\nInput data type:", X_train_tensor.dtype)
print("Label data type:", y_train_tensor.dtype)


Training input shape: torch.Size([1600, 200])
Testing input shape: torch.Size([400, 200])
Training labels shape: torch.Size([1600])
Testing labels shape: torch.Size([400])

Input data type: torch.int64
Label data type: torch.int64



## Task 7: Build the RNN Model


In [8]:

import torch.nn as nn

# Define the RNN model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # RNN layer
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Linear output layer
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)

        # Use the final hidden state for classification
        last_hidden = hidden[-1]
        return self.fc(last_hidden)


# Get vocabulary size from the tokenizer
vocab_size = encoding.n_vocab

# Create the model
model = RNNClassifier(vocab_size)

print(model)


RNNClassifier(
  (embedding): Embedding(100277, 64)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)



## Task 8: Train the RNN Model


In [9]:

from torch.utils.data import TensorDataset, DataLoader

# Create the training dataset and batches
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {average_loss:.4f}")


Epoch [1/5], Loss: 0.7038
Epoch [2/5], Loss: 0.6782
Epoch [3/5], Loss: 0.6565
Epoch [4/5], Loss: 0.6290
Epoch [5/5], Loss: 0.5861



## Task 9: Evaluate the RNN Model and Calculate Accuracy


In [10]:

# Evaluate the trained RNN model
model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)

# Calculate test accuracy
correct = (predictions == y_test_tensor).sum().item()
accuracy = correct / len(y_test_tensor) * 100

print("Total test reviews:", len(y_test_tensor))
print("Correct predictions:", correct)
print(f"Test Accuracy: {accuracy:.2f}%")


Total test reviews: 400
Correct predictions: 203
Test Accuracy: 50.75%



## Task 10: Test the RNN on Custom Reviews


In [11]:

# Three new movie reviews
custom_reviews = [
    "This movie was fantastic and I really enjoyed it.",
    "The film was boring and the acting was terrible.",
    "The story was interesting, but the ending was disappointing."
]

# Predict sentiment for each review
model.eval()

with torch.no_grad():
    for review in custom_reviews:
        tokens = encoding.encode(review, disallowed_special=())
        tokens = tokens[:MAX_LENGTH]
        tokens = tokens + [0] * (MAX_LENGTH - len(tokens))

        input_tensor = torch.tensor([tokens], dtype=torch.long)

        output = model(input_tensor)
        prediction = torch.argmax(output, dim=1).item()

        sentiment = "Positive" if prediction == 1 else "Negative"

        print("Review:", review)
        print("Predicted sentiment:", sentiment)
        print()


Review: This movie was fantastic and I really enjoyed it.
Predicted sentiment: Positive

Review: The film was boring and the acting was terrible.
Predicted sentiment: Positive

Review: The story was interesting, but the ending was disappointing.
Predicted sentiment: Positive

